In [ ]:
import os
import simo
from simo import load_data, process_anndata, find_marker, alignment_1, assign_coord_1
from simo.regulation import regulation_analysis, spatial_regulation
from simo import sdplot, sfplot, cor_plot, module_vilolin_plot, module_dot_plot, module_pca_plot, module_spatial_plot, module_heatmap_plot, spatial_lineplot, plot_3d
from simo.helper import extract_reduction
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import itertools
from tqdm import tqdm
import random
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import issparse
import scipy
from scanpy import AnnData
import time

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import random
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

set_seed(0)

In [ ]:
#127/9127 cells matched
sc_adata = sc.read('../data/processed/mousep22_rna.h5ad')
atac_adata = sc.read('../data/processed/mousep22_atac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(sc_adata, n_comps=30)
sc_adata.obsm['reduction'] = sc_adata.obsm['X_pca'].copy()
sc_adata.obs['type'] = sc_adata.obs['celltype'].copy()
atac_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * atac_adata.shape[0]})
expected_num.index = atac_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=atac_adata, adata2=sc_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=atac_adata, adata2=sc_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names].values
adata_new.obs['ATAC_cluster'] = sc_adata.obs['ATAC_clusters'][adata_new.obs_names].values

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']].values
adata_new.obs['ATAC_cluster_2'] = atac_adata.obs['ATAC_clusters'][adata_new.obs['SpotID']].values

In [ ]:
adata_new.write('../results/paired_SIMO_p22.h5ad')

In [ ]:
#29/2500 cells matched
sc_adata = sc.read('../data/processed/human_rna.h5ad')
atac_adata = sc.read('../data/processed/human_atac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(sc_adata, n_comps=30)
sc_adata.obsm['reduction'] = sc_adata.obsm['X_pca'].copy()
sc_adata.obs['type'] = sc_adata.obs['celltype'].copy()
atac_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * atac_adata.shape[0]})
expected_num.index = atac_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=atac_adata, adata2=sc_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=atac_adata, adata2=sc_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names].values
adata_new.obs['ATAC_cluster'] = sc_adata.obs['ATAC_clusters'][adata_new.obs_names].values

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']].values
adata_new.obs['ATAC_cluster_2'] = atac_adata.obs['ATAC_clusters'][adata_new.obs['SpotID']].values

In [ ]:
adata_new.write('../results/paired_SIMO_human.h5ad')

In [ ]:
#127/9127 cells matched
sc_adata = sc.read('../data/processed/mousep22_h3k27ac_rna.h5ad')
atac_adata = sc.read('../data/processed/mousep22_h3k27ac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(sc_adata, n_comps=30)
sc_adata.obsm['reduction'] = sc_adata.obsm['X_pca'].copy()
sc_adata.obs['type'] = sc_adata.obs['celltype'].copy()
atac_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * atac_adata.shape[0]})
expected_num.index = atac_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=atac_adata, adata2=sc_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=atac_adata, adata2=sc_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names]
adata_new.obs['H3K27ac_cluster'] = sc_adata.obs['H3K27ac_clusters'][adata_new.obs_names]

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']]
adata_new.obs['H3K27ac_cluster_2'] = atac_adata.obs['H3K27ac_clusters'][adata_new.obs['SpotID']]

In [ ]:
adata_new.write('../results/paired_SIMO_h3k27ac.h5ad')

In [ ]:
# set sc as spatial reference

In [ ]:
#127/9127 cells matched
sc_adata = sc.read('../data/processed/mousep22_rna.h5ad')
atac_adata = sc.read('../data/processed/mousep22_atac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(atac_adata, n_comps=30)
atac_adata.obsm['reduction'] = atac_adata.obsm['X_pca'].copy()
atac_adata.obs['type'] = atac_adata.obs['celltype'].copy()
sc_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * sc_adata.shape[0]})
expected_num.index = sc_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=sc_adata, adata2=atac_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=sc_adata, adata2=atac_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names].values
adata_new.obs['ATAC_cluster'] = sc_adata.obs['ATAC_clusters'][adata_new.obs_names].values

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']].values
adata_new.obs['ATAC_cluster_2'] = atac_adata.obs['ATAC_clusters'][adata_new.obs['SpotID']].values

In [ ]:
adata_new.write('../results/paired_SIMO_p22_verse.h5ad')

In [ ]:
#29/2500 cells matched
sc_adata = sc.read('../data/processed/human_rna.h5ad')
atac_adata = sc.read('../data/processed/human_atac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(atac_adata, n_comps=30)
atac_adata.obsm['reduction'] = atac_adata.obsm['X_pca'].copy()
atac_adata.obs['type'] = atac_adata.obs['celltype'].copy()
sc_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * sc_adata.shape[0]})
expected_num.index = sc_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=sc_adata, adata2=atac_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=sc_adata, adata2=atac_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names].values
adata_new.obs['ATAC_cluster'] = sc_adata.obs['ATAC_clusters'][adata_new.obs_names].values

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']].values
adata_new.obs['ATAC_cluster_2'] = atac_adata.obs['ATAC_clusters'][adata_new.obs['SpotID']].values

In [ ]:
adata_new.write('../results/paired_SIMO_human_verse.h5ad')

In [ ]:
#127/9127 cells matched
sc_adata = sc.read('../data/processed/mousep22_h3k27ac_rna.h5ad')
atac_adata = sc.read('../data/processed/mousep22_h3k27ac.h5ad')

sc_adata.obs['celltype'] = 'celltype'
atac_adata.obs['celltype'] = 'celltype'

In [ ]:
atac_to_genes = [atac[5:] for atac in list(atac_adata.var_names)]
atac_adata.var_names = atac_to_genes.copy()

shared_genes = np.intersect1d(atac_adata.var_names, sc_adata.var_names)

atac_adata = atac_adata[:, shared_genes].copy()
sc_adata   = sc_adata[:, shared_genes].copy()

In [ ]:
sc.tl.pca(atac_adata, n_comps=30)
atac_adata.obsm['reduction'] = atac_adata.obsm['X_pca'].copy()
atac_adata.obs['type'] = atac_adata.obs['celltype'].copy()
sc_adata.obs['type'] = 'uniform_region'
expected_num = pd.DataFrame({'cell_num': [1] * sc_adata.shape[0]})
expected_num.index = sc_adata.obs_names

sc_adata = process_anndata(sc_adata)
atac_adata = process_anndata(atac_adata)

alignment_result = alignment_1(adata1=sc_adata, adata2=atac_adata, alpha=0.1, aware_st=False, aware_sc=True)

spatial_assignment = assign_coord_1(adata1=sc_adata, adata2=atac_adata, out_data=alignment_result, non_zero_probabilities=True, no_repeated_cells=True, expected_num=expected_num, pos_random=False)

adata_new = sc_adata[spatial_assignment['cell'], :]
spatial_assignment = spatial_assignment.set_index('cell')
adata_new.obs[['SpotID', 'x_simo', 'y_simo']] = spatial_assignment[['spot', 'Cell_xcoord', 'Cell_ycoord']]
adata_new.obs['SpotID'] = adata_new.obs['SpotID'].astype(str)
adata_new.var_names = [item.upper() for item in adata_new.var_names]

In [ ]:
adata_new.obs['RNA_cluster'] = sc_adata.obs['RNA_clusters'][adata_new.obs_names]
adata_new.obs['H3K27ac_cluster'] = sc_adata.obs['H3K27ac_clusters'][adata_new.obs_names]

adata_new.obs['RNA_cluster_2'] = atac_adata.obs['RNA_clusters'][adata_new.obs['SpotID']]
adata_new.obs['H3K27ac_cluster_2'] = atac_adata.obs['H3K27ac_clusters'][adata_new.obs['SpotID']]

In [ ]:
adata_new.write('../results/paired_SIMO_h3k27ac_verse.h5ad')